In [19]:
import datetime
import pytz

edt = pytz.timezone("America/New_York")
today_edt = datetime.datetime.now(edt).date()

print(today_edt)

2026-05-14


In [38]:
from nba_api.stats.endpoints import scoreboardv3
from datetime import timedelta
import pandas as pd

def getUpcomingGames(max_lookahead=3):
    totalGames = []
    query_date = today_edt
    for i in range(max_lookahead):
        query_date = query_date + timedelta(days=1)
        data = scoreboardv3.ScoreboardV3(game_date=query_date)
        games = data.get_data_frames()[1]

        if games.empty:
            continue
        games.drop(columns=["period", "gameClock", "regulationPeriods"], inplace=True)
        games = games[games["ifNecessary"] != True]
        totalGames.append(games)
    df_games = pd.concat(totalGames, ignore_index=True)
    return df_games
        


In [42]:
df_games = getUpcomingGames()
df_games

,gameId,gameCode,gameStatus,gameStatusText,gameTimeUTC,gameEt,seriesGameNumber,gameLabel,gameSubLabel,seriesText,ifNecessary,seriesConference,poRoundDesc,gameSubtype,isNeutral
0,0042500206,20260515/DETCLE,1,7:00 pm ET,2026-05-15T23:00:00Z,2026-05-15T19:00:00Z,Game 6,East Conf. Semifinals,Game 6,CLE leads 3-2,False,East,Conf. Semifinals,,False
1,0042500236,20260515/SASMIN,1,9:30 pm ET,2026-05-16T01:30:00Z,2026-05-15T21:30:00Z,Game 6,West Conf. Semifinals,Game 6,SAS leads 3-2,False,West,Conf. Semifinals,,False


In [43]:
df_games.to_csv("data/upcoming_games.csv", index=False, encoding='utf-8')

In [44]:
# Update Postgres DB
from sqlalchemy import create_engine
from config import Config

engine = create_engine(Config.SQLALCHEMY_DATABASE_URI)

In [45]:
df_games.columns = df_games.columns.str.lower()

In [46]:
from sqlalchemy import text
df_games.to_sql("upcoming_games", engine, schema="nba_data", if_exists="replace", index=False)
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE nba_data.upcoming_games ADD PRIMARY KEY (gameId);"))
    conn.commit()